# Model output

In [ ]:
import sys
sys.path.append('../..')
sys.path.append('..')

In [ ]:
import os
import torch
import numpy as np
import pathlib

from modis.model import MODIS
from modis.utils.config import load_config
from modis.utils.data import get_dataloaders, summarize_dataset, get_samples_from_dataloader
from modis.utils.plots import plot_training_log, get_class_per_modality_colors, plot_2d_projection, plot_3d_projection, plot_confusion_matrix

from src.intersim_dataset import get_datasets

In [ ]:
# Adjust these variables
checkpoint_path = pathlib.Path('../saved/checkpoints/intersim_2_delta/intersim/20250717_104628/')
config_file = pathlib.Path('../config/intersim/intersim.yaml')
use_best = True
save_plot = False

# Don't touch these
config = load_config(config_file)
checkpoint_file = checkpoint_path / f"{'checkpoint_best.pth' if use_best else 'checkpoint_latest.pth'}"
num_modalities = len(config.modalities)
modality_names = [item.name for item in config.modalities]

## Plot training log

In [ ]:
plot_training_log(
    training_mode = config.training_mode,
    modality_names = modality_names,
    checkpoint_path = checkpoint_path,
    use_best = use_best,
    save_plot = save_plot
)

## Load the model

In [ ]:
model = MODIS(config)
model.load_from_checkpoint(checkpoint_file)

In [ ]:
model.print_model_params()

## Load the data

In [ ]:
datasets = get_datasets(
    dataset_name = 'intersim_2_delta',
    pairing = 'unpaired',
    split = 'test',
    data_path = '../data',
    include_sample_ids = False
)

# Make semisupervised datasets
# from modis.utils.data import SemiSupervisedDataset
# datasets = [SemiSupervisedDataset(dataset, labeled_ratio=0.0) for dataset in datasets]

dataloaders = get_dataloaders(datasets, batch_size=config.batch_size, drop_last=False, shuffle=True)
summarize_dataset(dataloaders, modality_names=modality_names)

In [ ]:
# Get a list of samples per modality
x, y = list(zip(*[get_samples_from_dataloader(dataloader, num_samples=None, device=config.device) for dataloader in dataloaders]))

# Remove unlabeled samples
if config.training_mode == 'semisupervised':
    x = list(x)
    y = list(y)
    for i in range(num_modalities):
        labeled_mask = torch.tensor([True if label != -1 else False for label in y[i]])
        x[i] = x[i][labeled_mask]
        y[i] = y[i][labeled_mask]

total_labeled_samples = sum([len(ds) for ds in y])
if total_labeled_samples == 0:
    raise Exception("At least some labeled samples are needed")

# Latents
modal_latents = [model.get_latents(x[i], input_modality=i) for i in range(num_modalities)]
latents = torch.concat(modal_latents, dim=0).cpu().numpy()

# Labels
class_labels = torch.concat(y, dim=0).cpu().numpy()
modality_labels = torch.concat([torch.full((x[i].shape[0],), i) for i in range(num_modalities)], dim=0).numpy()
class_per_modality_labels = [cl + mi*config.num_classes for mi, modality_class_labels in enumerate(y) for cl in modality_class_labels.cpu().numpy()]  ####### Assuming equal number of classes per modality

# Colors
modality_colors = get_class_per_modality_colors(num_modalities=1, num_classes=num_modalities)
class_colors = get_class_per_modality_colors(num_modalities=1, num_classes=config.num_classes)
class_per_modality_colors = get_class_per_modality_colors(num_modalities=num_modalities, num_classes=config.num_classes)

# Names
modality_names = [f"{config.modalities[i].name}" for i in range(num_modalities)]
class_names = [i for i in range(config.num_classes)]
class_per_modality_names = [f"{config.modalities[mi].name}_{ci}" for mi in range(num_modalities) for ci in range(config.num_classes)]

## Plot projections

In [ ]:
plot_2d_projection(
    technique = 'pca',
    data = latents,
    labels = class_per_modality_labels,
    labels_colors = class_per_modality_colors,
    labels_names = class_per_modality_names,
    standardize = False,
    checkpoint_path = checkpoint_path,
    is_best = use_best,
    save_plot = save_plot,
    text_size = 26
)

In [ ]:
plot_3d_projection(
    technique = 'pca',
    data = latents,
    labels = class_per_modality_labels,
    labels_colors = class_per_modality_colors,
    labels_names = class_per_modality_names,
    standardize = False,
    checkpoint_path = checkpoint_path,
    is_best = use_best,
    save_plot = save_plot
)

## Confusion matrices

In [ ]:
class_labels_pred = model.discriminator.predict(torch.tensor(latents).to(config.device)).cpu().numpy()
aux_modality_class_labels_pred = np.array(list(map(lambda i: class_labels_pred[i] + modality_labels[i]*config.num_classes, range(len(class_labels_pred)))), dtype=np.int32)  # using the modality of origin

In [ ]:
plot_confusion_matrix(
    class_labels,
    class_labels_pred,
    performance_metrics = True,
    checkpoint_path = checkpoint_path,
    is_best = use_best,
    save_plot = save_plot,
    figsize = (16, 10),
    text_size = 25
)

In [ ]:
plot_confusion_matrix(
    class_per_modality_labels,
    aux_modality_class_labels_pred,
    performance_metrics = True,
    checkpoint_path = checkpoint_path,
    is_best = use_best,
    save_plot = save_plot,
    figsize = (40, 20),
    text_size = 25,
    filename_suffix = 'class_per_modality'
)

In [ ]:
for i in range(num_modalities):
    plot_confusion_matrix(
        class_labels[modality_labels == i],
        class_labels_pred[modality_labels == i],
        performance_metrics = True,
        checkpoint_path = checkpoint_path,
        is_best = use_best,
        save_plot = save_plot,
        figsize = (16, 10),
        text_size = 25,
        filename_suffix = f'modality_{modality_names[i]}'
    )

## Reconstruction

**Important**:
- The dataset must be paired so that the reconstructions are comparable
- Very memory intensive

In [ ]:
translations = {}
originals = {}
for input_modality in range(num_modalities):
    translations[input_modality] = {}
    for target_modality in range(num_modalities):
        # translations[input_modality][target_modality] =  model.translate(x[input_modality], input_modality=input_modality, target_modality=target_modality)

        with torch.no_grad():
            translation_latents = model.get_latents(x[input_modality], input_modality=input_modality)
            translations[input_modality][target_modality] = model.variational_autoencoders[target_modality].decode(translation_latents).cpu().numpy()

In [ ]:
mse_matrix = np.zeros((num_modalities, num_modalities))
for ii in range(num_modalities):
    for it in range(num_modalities):
        mse_matrix[ii, it] = np.mean((translations[ii][it] - x[it].cpu().numpy())**2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 7))

text_size = 30
vmin, vmax = 0, 0.1
ax = sns.heatmap(mse_matrix, annot=True, fmt=".4f", cmap="coolwarm", linewidths=0.5, cbar=True, annot_kws={"size": text_size}, vmin=vmin, vmax=vmax)

plt.xlabel("output modality", fontsize=text_size)
plt.ylabel("input modality", fontsize=text_size)

ticks = ["1", "2", "3"]
ax.set_xticklabels(ticks, fontsize=text_size)
ax.set_yticklabels(ticks, fontsize=text_size)

# plt.title("MSE Heatmap", fontsize=text_size)

# Customize colorbar
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=text_size)

plt.show()
# plt.savefig(f'/home/daniel/Desktop/dev/mk_program/mk2_project/dataset_models/intersim/saved/reports/mse_matrix.svg', format='svg', bbox_inches='tight')
# plt.close()

In [ ]:
mse = []
for input_modality in range(num_modalities):
    for target_modality in range(num_modalities):
        if input_modality != target_modality: continue
        print(input_modality, target_modality)
        mse.append( np.mean((translations[input_modality][target_modality] - x[target_modality].cpu().numpy())**2) )
print(f"MSE: {np.mean(mse)}")

In [ ]:
mse

In [ ]:
def all_pairs_mse(X, X_hat):
    differences = X[:, None, :] - X_hat[None, :, :]  # Broadcast to [n_samples_X, n_samples_X_hat, n_features]
    mse_matrix = torch.mean(differences ** 2, dim=2)  # Averaging over the feature axis (dim=2)
    return mse_matrix.mean().item()

In [ ]:
all_pairs_mse(x[0].cpu(), translations[0][0])